In [7]:
import tensorflow as tf
import tensorflow_decision_forests as tfdf
import numpy as np
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras import Model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
import matplotlib.pyplot as plt
import pandas as pd

from lime.lime_tabular import LimeTabularExplainer
import shap
from collections import defaultdict

In [2]:
wine_qt_path = '../data/WineQT.csv'
RANDOM_SEED = 492
wine_df = pd.read_csv(wine_qt_path)

In [3]:
y_series = wine_df['quality']
y = pd.DataFrame(y_series, columns=['quality'])
features = [col for col in wine_df.columns if col != 'Id' and col != 'quality']
X = wine_df[features]
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_series)
y_encoded = pd.DataFrame(y_encoded, columns=['quality'])

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=RANDOM_SEED)

In [5]:
X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)


In [ ]:
# Define the MLP model for classification
class MLPClassification(tf.keras.Model):
    def __init__(self, input_dim, hidden_dim, dropout_rate=0.1, num_classes=6):
        super(MLPClassification, self).__init__()
        self.dense1 = Dense(hidden_dim, activation='relu')
        self.dropout1 = Dropout(dropout_rate)
        self.dense2 = Dense(hidden_dim, activation='relu')
        self.dropout2 = Dropout(dropout_rate)
        self.dense3 = Dense(num_classes, activation='softmax')
    
    def call(self, inputs, training=False):
        x = self.dense1(inputs)
        x = self.dropout1(x, training=training)
        x = self.dense2(x)
        x = self.dropout2(x, training=training)
        x = self.dense3(x)
        return x

In [ ]:
# DropConnectDense layer
class DropConnectDense(tf.keras.layers.Layer):
    def __init__(self, units, dropout_rate=0.1, **kwargs):
        super(DropConnectDense, self).__init__(**kwargs)
        self.units = units
        self.dropout_rate = dropout_rate
        self.dense = Dense(units)

    def build(self, input_shape):
        self.kernel = self.add_weight("kernel", shape=[input_shape[-1], self.units])
        self.bias = self.add_weight("bias", shape=[self.units])

    def call(self, inputs, training=False):
        if training:
            weights = self.dense.kernel * tf.keras.backend.random_binomial(
                shape=self.dense.kernel.shape, p=1 - self.dropout_rate)
            output = tf.keras.backend.dot(inputs, weights)
            if self.dense.use_bias:
                output = tf.keras.backend.bias_add(output, self.dense.bias)
            return self.dense.activation(output)
        else:
            return self.dense(inputs)


In [ ]:
# Define the MLP model with DropConnect for classification
class MLPDropConnectClassification(tf.keras.Model):
    def __init__(self, input_dim, hidden_dim, dropout_rate=0.1, num_classes=6):
        super(MLPDropConnectClassification, self).__init__()
        self.dense1 = DropConnectDense(hidden_dim, dropout_rate)
        self.dense2 = DropConnectDense(hidden_dim, dropout_rate)
        self.dense3 = Dense(num_classes, activation='softmax')
    
    def call(self, inputs, training=False):
        x = self.dense1(inputs, training=training)
        x = self.dense2(x, training=training)
        x = self.dense3(x)
        return x


In [ ]:
# Model with uncertainty handling
class ModelWithUncertainty(Model):
    def __init__(self, input_dim, hidden_dim, dropout_rate=0.1, method='mc_dropout', num_classes=6):
        super(ModelWithUncertainty, self).__init__()
        self.method = method
        if method == 'mc_dropout':
            self.model = MLPClassification(input_dim, hidden_dim, dropout_rate, num_classes)
        elif method == 'dropconnect':
            self.model = MLPDropConnectClassification(input_dim, hidden_dim, dropout_rate, num_classes)
        else:
            raise ValueError("Method should be 'mc_dropout' or 'dropconnect'")
    
    def call(self, inputs, training=False):
        return self.model(inputs, training=training)
    
    def predict_with_uncertainty(self, inputs, num_samples):
        return predict_with_uncertainty(self.model, inputs, num_samples)

    def explain_lime(self, instance, data, num_features):
        return explain_lime(self.model, instance, data, num_features=num_features)

    def explain_shap(self, instance, data):
        return explain_shap(self.model, instance, data)


In [ ]:
# Function to predict with uncertainty
def predict_with_uncertainty(model, inputs, num_samples=10):
    inputs = tf.expand_dims(inputs, axis=0)  # Add batch dimension
    predictions = []
    lime_explanations = []
    shap_explanations = []
    for _ in range(num_samples):
        prediction = model(inputs, training=True)
        predictions.append(prediction)
        lime_explanation = explain_lime(model, inputs[0], X_train_scaled, num_features)
        shap_explanation = explain_shap(model, inputs[0], X_train_scaled)
        lime_explanations.append(lime_explanation)
        shap_explanations.append(shap_explanation)
    predictions = tf.stack(predictions, axis=0)
    prediction_mean = tf.reduce_mean(predictions, axis=0)
    prediction_std = tf.math.reduce_std(predictions, axis=0)
    return prediction_mean.numpy(), prediction_std.numpy(), lime_explanations, shap_explanations

# Function to explain with LIME
def explain_lime(model, instance, data, num_features):
    explainer = LimeTabularExplainer(
        training_data=np.array(data),
        feature_names=[f'feature_{i}' for i in range(data.shape[1])],
        mode='classification'
    )
    explanation = explainer.explain_instance(instance.numpy(), model.predict, num_features=num_features)
    return explanation

# Function to explain with SHAP
def explain_shap(model, instance, data):
    explainer = shap.KernelExplainer(model.predict, data)
    shap_values = explainer.shap_values(instance.numpy())
    return shap_values


In [ ]:
num_classes = 6